In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split

def generar_datos_produccion(csv_path):
    print(f"📂 Cargando {csv_path}...")
    df_full = pd.read_csv(csv_path)
    
    # 1. Recreamos tu split original exacto para aislar el Test
    # Usamos random_state=42 para que los datos caigan exactamente igual que en tus pruebas
    _, test_full = train_test_split(df_full, test_size=0.2, random_state=42)
    
    print("✂️ Extrayendo 12 productos por categoría del conjunto de Test...")
    
    # 2. Seleccionar 12 productos aleatorios de cada categoría dentro del Test
    # Usamos replace=False para no repetir y manejamos el caso de que alguna categoría tenga menos de 12
    muestras_dashboard = test_full.groupby('category', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), 12), random_state=42)
    )
    
    # 3. Guardar estos productos intocables para usar en Streamlit
    ruta_dashboard = "dashboard_test_samples.csv"
    muestras_dashboard.to_csv(ruta_dashboard, index=False)
    print(f"✅ ¡Guardados {len(muestras_dashboard)} productos para el Dashboard en '{ruta_dashboard}'!")
    
    # 4. Crear el dataset "100%" eliminando SOLO estos 12 productos por categoría
    # Usamos los índices para borrarlos del dataframe original completo
    indices_a_borrar = muestras_dashboard.index
    df_train_produccion = df_full.drop(index=indices_a_borrar)
    
    print(f"📊 Dataset Original: {len(df_full)} filas")
    print(f"📊 Dataset Producción (100% - Muestras): {len(df_train_produccion)} filas")
    
    return df_train_produccion, muestras_dashboard

# Ejecutamos la función
df_train_produccion, df_dashboard = generar_datos_produccion("completo_con_urls.csv")

📂 Cargando completo_con_urls.csv...
✂️ Extrayendo 12 productos por categoría del conjunto de Test...
✅ ¡Guardados 120 productos para el Dashboard en 'dashboard_test_samples.csv'!
📊 Dataset Original: 6442 filas
📊 Dataset Producción (100% - Muestras): 6322 filas


C:\Users\diego\AppData\Local\Temp\ipykernel_7620\524128643.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  muestras_dashboard = test_full.groupby('category', group_keys=False).apply(


In [35]:
df_train_produccion.to_csv("../../../Datasets/evaluacion4_produccion.csv", index=False)

In [23]:
evaluacion4 = pd.read_csv("../../../Datasets/evaluacion4.csv")

completo = pd.read_csv("../Extractor_csv/amazon_specs_completo.csv")

In [26]:
completo['category'].value_counts()

category
Office, Printing & Power        1713
Audio & Media Systems            927
Accessories                      756
Peripherals & Input              641
PC Components (Core)             543
Computers & Gaming               480
Cameras, Photography & Video     451
Displays & Mounting              424
Networking & Smart Home          285
Mobile Devices                   222
Name: count, dtype: int64

In [5]:
evaluacion4['category'].value_counts()

category
Office, Printing & Power        1713
Audio & Media Systems            927
Accessories                      756
Peripherals & Input              641
PC Components (Core)             543
Computers & Gaming               480
Cameras, Photography & Video     451
Displays & Mounting              424
Networking & Smart Home          285
Mobile Devices                   222
Name: count, dtype: int64

In [25]:
# 1. Creamos la condición exacta de las filas que queremos ELIMINAR
condicion_eliminar = (completo['category'] == 'Networking & Smart Home') & (completo['subtype'].isin(['rack_component', 'controller']))

# 2. Nos quedamos con todo el DataFrame EXCEPTO esas filas (usando ~)
completo = completo[~condicion_eliminar].copy()

# Opcional: Comprobación rápida para ver que ha funcionado
print("Subtipos restantes en Networking:")
print(completo[completo['category'] == 'Networking & Smart Home']['subtype'].unique())

Subtipos restantes en Networking:
['smart_tag' 'wifi_extender' 'router' 'mesh_system' 'switch' 'modem'
 'wifi_adapter' 'video_doorbell' 'access_point' 'smart_sensor'
 'smart_alarm']


In [27]:
completo["brand_subtype"] = completo["brand"] + "_" + completo["subtype"]

In [28]:
completo.head()

,category,subtype,market_tier,condition,is_premium_brand,tech_generation,brand,confidence,printer_tech,is_color,...,is_best_seller,is_sponsored,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,log_original_price,log_purchased_last_month,log_total_reviews,brand_subtype
0,PC Components (Core),ssd_internal,Mainstream,New,False,Current-Gen,OWC,0.95,NaN,NaN,...,No Badge,Sponsored,1,0,0,0.0,5.484755,0.000000,5.883322,OWC_ssd_internal
1,PC Components (Core),ssd_internal,Mainstream,New,True,Current-Gen,OWC,0.95,NaN,NaN,...,No Badge,Sponsored,1,0,0,0.0,3.783962,0.000000,5.624018,OWC_ssd_internal
2,"Office, Printing & Power",toner_ink,Mainstream,New,True,Current-Gen,HP,0.95,Inkjet,False,...,Best Seller,Organic,0,1,0,0.0,3.607941,10.819798,11.523598,HP_toner_ink
3,"Office, Printing & Power",toner_ink,Mainstream,New,False,Current-Gen,HP,0.95,Inkjet,True,...,No Badge,Organic,0,0,0,0.0,3.804215,10.819798,10.986868,HP_toner_ink
4,Audio & Media Systems,headphones,Mainstream,New,False,Current-Gen,Sony,0.95,NaN,True,...,Best Seller,Organic,0,0,0,0.0,2.355178,9.210440,11.598433,Sony_headphones


In [29]:
df_url = pd.read_csv("../../../Datasets/amazon_products_sales_data_uncleaned.csv")
df_url.head()

,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6 out of 5 stars,375,300+ bought in past month,89.68,basic variant price: 2.4GHz,$159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3 out of 5 stars,"2,457",6K+ bought in past month,9.99,basic variant price: nan,$15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6 out of 5 stars,"3,044",2K+ bought in past month,314.00,basic variant price: nan,$349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6 out of 5 stars,"35,882",10K+ bought in past month,NaN,basic variant price: $162.24,No Discount,Best Seller,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,/Apple-Cancellation-Transparency-Personalized-...,2025-08-21 11:14:29
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8 out of 5 stars,"28,988",10K+ bought in past month,NaN,basic variant price: $72.74,No Discount,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,/Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref...,2025-08-21 11:14:29


In [30]:
import pandas as pd

# 1. Quitamos los duplicados, pero indicamos keep='last' para quedarnos con el scrapeo más reciente
df_url_limpio = df_url[['title', 'image_url', 'product_url']].drop_duplicates(subset=['title'], keep='last')

# 2. Hacemos el cruce (Left Join) con tu dataframe completo
df_final = pd.merge(
    completo,
    df_url_limpio, 
    left_on='original_title', 
    right_on='title', 
    how='left'
)

# 3. Limpiamos la columna 'title' sobrante que viene de df_url_limpio
df_final = df_final.drop(columns=['title'])

# Comprobación de seguridad
print(f"✅ Cruce completado quedándonos con la última URL scrapeada.")
print(f"📊 Filas originales: {len(completo)}")
print(f"📊 Filas finales: {len(df_final)}")

✅ Cruce completado quedándonos con la última URL scrapeada.
📊 Filas originales: 6442
📊 Filas finales: 6442


In [32]:
df_final.drop(columns=['model_used', 'original_row_id', 'error_log'], inplace=True)

In [33]:
df_final.to_csv("completo_con_urls.csv", index=False)

In [36]:
aux = pd.read_csv("../../../Codigo/dashboard_test_samples.csv")

In [ ]:
aux = aux.rename(columns={'image_url': 'product_image_url'})